# Stage 2 (~1M trainable parameters): PANNs CNN14 + larger residual embedding adapter

This notebook continues the staged ICBHI experiment after Stage 1 (~100k parameters).

- Task: binary ICBHI classification, **Normal (0)** vs **Abnormal (1)**.
- The pretrained CNN14 backbone stays frozen and in evaluation mode.
- The adapter bottleneck increases from 23 (Stage 1) to **243** (Stage 2).
- Target trainable parameter count: **1,001,717** (adapter + binary classifier).
- The Stage 1 best checkpoint is used to initialize compatible backbone and classifier weights; the larger adapter is initialized fresh because its tensor shapes differ.
- Train/validation splitting remains patient-level and uses only official training patients.
- The official test split is not used for checkpoint selection; best checkpoint is selected by validation HS-score, and the last epoch is saved separately.



In [1]:
# 1. Imports and reproducibility
import os
import random
import json
import importlib.util
import warnings
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import librosa
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import confusion_matrix, accuracy_score


D:\.JKU CS courses\Master Thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#from audioset_tagging_cnn_master.pytorch.models import Cnn14
SRC_DIR = Path.cwd()
PYTORCH_DIR = SRC_DIR / "audioset_tagging_cnn_master" / "pytorch"
MODELS_PATH = PYTORCH_DIR / "models.py"
LOCAL_UTILS_PATH = PYTORCH_DIR / "pytorch_utils.py"

assert MODELS_PATH.is_file(), f"Missing file: {MODELS_PATH}"
assert LOCAL_UTILS_PATH.is_file(), f"Missing file: {LOCAL_UTILS_PATH}"

# Make the local helper available under the name expected by AudioSet.
utils_spec = importlib.util.spec_from_file_location(
    "pytorch_utils", LOCAL_UTILS_PATH
)
utils_module = importlib.util.module_from_spec(utils_spec)
sys.modules["pytorch_utils"] = utils_module


# Execute the local helper module
utils_spec.loader.exec_module(utils_module)

print("Local helper loaded:", utils_module.__file__)
print("Has forward:", hasattr(utils_module, "forward"))

# Load models.py directly, without executing the package __init__.py
models_spec = importlib.util.spec_from_file_location(
    "audioset_cnn14_models", MODELS_PATH
)
models_module = importlib.util.module_from_spec(models_spec)
sys.modules[models_spec.name] = models_module
models_spec.loader.exec_module(models_module)

Cnn14 = models_module.Cnn14
print("CNN14 imported successfully:", Cnn14)



SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Local helper loaded: D:\.JKU CS courses\Master Thesis\src\audioset_tagging_cnn_master\pytorch\pytorch_utils.py
Has forward: True
CNN14 imported successfully: <class 'audioset_cnn14_models.Cnn14'>
Device: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU


In [3]:
# 2. Paths and hyperparameters 
AUDIO_DIR = Path("../datasets/raw/ICBHI_final_database/audios")
METADATA_PATH = Path("../datasets/split_metadata.csv")
PRETRAINED_CHECKPOINT = Path("Cnn14_mAP=0.431.pth")
OUTPUT_DIR = Path("../models/cnn14_stage2_1m_adapter")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_RATE = 16_000
SEGMENT_SECONDS = 10.0  # fixed input length for batching; segments are cropped using metadata start/end
BATCH_SIZE = 8
NUM_WORKERS = 0  # safe default on Windows/Jupyter
EPOCHS = 20
VAL_SIZE = 0.20
CLASSIFIER_LR = 3e-4
ADAPTER_LR = 1e-4
WEIGHT_DECAY = 1e-4
ADAPTER_DIM = 243  # yields 1,001,717 trainable parameters including the classifier
USE_AMP = False
RUN_FINAL_TEST_EVALUATION = False  # keep False while comparing training stages

STAGE1_BEST_CHECKPOINT = Path("../models/cnn14_stage1_100k_adapter/cnn14_best_val_hs.pt")
BEST_CHECKPOINT = OUTPUT_DIR / "cnn14_best_val_hs.pt"
LAST_CHECKPOINT = OUTPUT_DIR / "cnn14_last_epoch.pt"
RESULTS_PATH = OUTPUT_DIR / "test_results.json"

print("Audio directory:", AUDIO_DIR.resolve())
print("Metadata:", METADATA_PATH.resolve())
print("Pretrained checkpoint:", PRETRAINED_CHECKPOINT.resolve())
print("Stage 1 best checkpoint:", STAGE1_BEST_CHECKPOINT.resolve())

Audio directory: D:\.JKU CS courses\Master Thesis\datasets\raw\ICBHI_final_database\audios
Metadata: D:\.JKU CS courses\Master Thesis\datasets\split_metadata.csv
Pretrained checkpoint: D:\.JKU CS courses\Master Thesis\src\Cnn14_mAP=0.431.pth
Stage 1 best checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_stage1_100k_adapter\cnn14_best_val_hs.pt


In [4]:
# 3. Load and validate metadata
for required_path in (AUDIO_DIR, METADATA_PATH, PRETRAINED_CHECKPOINT):
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path does not exist: {required_path.resolve()}\n"
            "Update AUDIO_DIR, METADATA_PATH, and PRETRAINED_CHECKPOINT to match your local files."
        )

df = pd.read_csv(METADATA_PATH)
required_columns = {"source_file", "start", "end", "crackle", "wheeze", "split"}
missing = required_columns - set(df.columns)
if missing:
    raise ValueError(f"Metadata is missing required columns: {sorted(missing)}")

df = df.copy()
df["source_file"] = df["source_file"].astype(str).str.strip()
df["split"] = df["split"].astype(str).str.strip().str.lower()
split_map = {"train": "train", "training": "train", "test": "test", "testing": "test"}
unknown = sorted(set(df["split"]) - set(split_map))
if unknown:
    raise ValueError(f"Unexpected split values {unknown}; expected official train/test labels.")
df["split"] = df["split"].map(split_map)

def patient_id_from_source(source_file):
    stem = Path(source_file).stem
    if "_" not in stem:
        raise ValueError(
            f"Cannot infer patient ID from '{source_file}'. Expected ICBHI recording filename "
            "with patient ID before the first underscore."
        )
    return stem.split("_", 1)[0]

df["patient_id"] = df["source_file"].map(patient_id_from_source)
if df[list(required_columns)].isna().any().any():
    raise ValueError("Metadata contains missing values in required columns.")

official_train = df[df["split"] == "train"].copy()
official_test = df[df["split"] == "test"].copy()
if official_train.empty or official_test.empty:
    raise ValueError("Both official train and official test rows must be present.")

""" 
since there is data overlap in the official data split, the error << raise ValueError() >>
will be ignored and replaced with a warning message << warnings.warn() >>
"""
overlap = set(official_train.patient_id) & set(official_test.patient_id)
if overlap:
    warnings.warn(
        f"\n!!! WARNING DATA LEAKAGE !!!\nPatient leakage in metadata: {len(overlap)} patient(s) appear in "
        f"both official train and test, e.g. {sorted(overlap)[:10]}. Correct split_metadata.csv before training!"
    )

print(f"Official train: {len(official_train):,} segments / {official_train.patient_id.nunique()} patients")
print(f"Official test:  {len(official_test):,} segments / {official_test.patient_id.nunique()} patients")

Official train: 4,142 segments / 79 patients
Official test:  2,756 segments / 49 patients


C:\Users\arebr\AppData\Local\Temp\ipykernel_25332\2842253417.py:48: UserWarning: 
!!! WARNING DATA LEAKAGE !!!
Patient leakage in metadata: 2 patient(s) appear in both official train and test, e.g. ['156', '218']. Correct split_metadata.csv before training!
  warnings.warn(


In [5]:
# 4. Patient-level validation split from official training data only

splitter = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=SEED)

tr_idx, va_idx = next(splitter.split(
                                    X=np.zeros(len(official_train)),
                                    y=official_train["label"],
                                    groups=official_train["patient_id"]
                                    )
)

train_df = official_train.iloc[tr_idx].reset_index(drop=True)
val_df = official_train.iloc[va_idx].reset_index(drop=True)
test_df = official_test.reset_index(drop=True)  # Keep official test untouched

# Verify patient separation
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])

assert train_patients.isdisjoint(val_patients), "Train/validation leakage!"

# Binary labels: 0 = Normal, 1 = Abnormal
def binary_label(crackle, wheeze):
    return 0 if int(crackle) == 0 and int(wheeze) == 0 else 1

for frame in (train_df, val_df, test_df):
    frame["label"] = [
        binary_label(c, w)
        for c, w in zip(frame["crackle"], frame["wheeze"])
    ]

# Split sizes and patient counts
display(pd.DataFrame([
    {
        "split": "train",
        "segments": len(train_df),
        "patients": train_df["patient_id"].nunique()
    },
    {
        "split": "validation",
        "segments": len(val_df),
        "patients": val_df["patient_id"].nunique()
    },
    {
        "split": "official_test",
        "segments": len(test_df),
        "patients": test_df["patient_id"].nunique()
    }
]))

# Compare class distributions
for name, frame in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Official test", test_df)
]:
    print(f"\n{name} class counts:")
    print(frame["label"].value_counts().reindex([0, 1], fill_value=0))
    print(f"{name} class proportions:")
    print(frame["label"].value_counts(normalize=True).reindex(
        [0, 1], fill_value=0
    ).round(4))

print("\nNo patient overlap between train and validation.")


,split,segments,patients
0,train,3485,61
1,validation,657,18
2,official_test,2756,49



Train class counts:
label
0    1788
1    1697
Name: count, dtype: int64
Train class proportions:
label
0    0.5131
1    0.4869
Name: proportion, dtype: float64

Validation class counts:
label
0    275
1    382
Name: count, dtype: int64
Validation class proportions:
label
0    0.4186
1    0.5814
Name: proportion, dtype: float64

Official test class counts:
label
0    1579
1    1177
Name: count, dtype: int64
Official test class proportions:
label
0    0.5729
1    0.4271
Name: proportion, dtype: float64

No patient overlap between train and validation.


In [6]:
# 5. Audio dataset: crop each annotated segment, resample to 16 kHz, pad/trim to fixed duration
class ICBHIAudioDataset(Dataset):
    def __init__(self, frame, audio_dir):
        self.frame = frame.reset_index(drop=True).copy()
        self.audio_dir = Path(audio_dir)
        self.cache = {}

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        filename = row["source_file"]
        if not filename.lower().endswith(".wav"):
            filename += ".wav"
        wav_path = self.audio_dir / filename
        if not wav_path.exists():
            raise FileNotFoundError(f"Audio file not found: {wav_path}")

        if filename not in self.cache:
            audio, _ = librosa.load(wav_path, sr=SAMPLE_RATE, mono=True)
            self.cache[filename] = audio
        audio = self.cache[filename]

        start = max(0, int(float(row["start"]) * SAMPLE_RATE))
        end = min(len(audio), int(float(row["end"]) * SAMPLE_RATE))
        segment = audio[start:end]
        if len(segment) == 0:
            raise ValueError(f"Empty segment for {filename}: start={row['start']}, end={row['end']}")

        target_len = int(SAMPLE_RATE * SEGMENT_SECONDS)
        if len(segment) < target_len:
            segment = np.pad(segment, (0, target_len - len(segment)))
        else:
            segment = segment[:target_len]

        # Return waveform [time] and integer class label.
        return torch.tensor(segment, dtype=torch.float32), torch.tensor(int(row["label"]), dtype=torch.long)

train_ds = ICBHIAudioDataset(train_df, AUDIO_DIR)
val_ds = ICBHIAudioDataset(val_df, AUDIO_DIR)
test_ds = ICBHIAudioDataset(test_df, AUDIO_DIR)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
                          pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                        pin_memory=(device.type == "cuda"))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                         pin_memory=(device.type == "cuda"))

wave_batch, label_batch = next(iter(train_loader))
print("Waveform batch:", tuple(wave_batch.shape), "label batch:", tuple(label_batch.shape))

Waveform batch: (8, 160000) label batch: (8,)


In [7]:
# 6. Initialize CNN14 and add the Stage 2 (~1M parameter) residual adapter
# Load the original 527-class AudioSet model so checkpoint dimensions match.
backbone = Cnn14(
    sample_rate=16000,
    window_size=1024,
    hop_size=320,
    mel_bins=64,
    fmin=50,
    fmax=8000,
    classes_num=527,
)

checkpoint = torch.load(PRETRAINED_CHECKPOINT, map_location="cpu", weights_only=False)
state_dict = checkpoint["model"] if isinstance(checkpoint, dict) and "model" in checkpoint else checkpoint
missing, unexpected = backbone.load_state_dict(state_dict, strict=False)
print("Pretrained checkpoint loaded.")
print("Missing keys:", missing[:10], "..." if len(missing) > 10 else "")
print("Unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")

class CNN14BinaryClassifier(nn.Module):
    def __init__(self, cnn14_backbone, embedding_dim=2048, num_classes=2, adapter_dim=243):
        super().__init__()
        self.backbone = cnn14_backbone
        # Bottleneck MLP adapter: 2048 -> 243 -> 2048.
        # Its final layer is zero-initialized, so the adapter initially acts as an identity residual.
        self.embedding_adapter = nn.Sequential(
            nn.Linear(embedding_dim, adapter_dim),
            nn.ReLU(),
            nn.Linear(adapter_dim, embedding_dim),
        )
        nn.init.zeros_(self.embedding_adapter[-1].weight)
        nn.init.zeros_(self.embedding_adapter[-1].bias)
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, waveform):
        outputs = self.backbone(waveform, None)
        embedding = outputs["embedding"]
        adapted_embedding = embedding + self.embedding_adapter(embedding)
        logits = self.classifier(adapted_embedding)
        return logits

model = CNN14BinaryClassifier(backbone, adapter_dim=ADAPTER_DIM).to(device)

# Initialize compatible weights from Stage 1's best validation-HS checkpoint.
# The adapter is deliberately excluded because its dimensions changed (23 -> 243).
if not STAGE1_BEST_CHECKPOINT.exists():
    raise FileNotFoundError(
        f"Stage 1 best checkpoint not found: {STAGE1_BEST_CHECKPOINT.resolve()}\\n"
        "Run Stage 1 first, or correct STAGE1_BEST_CHECKPOINT in the configuration cell."
    )

stage1_payload = torch.load(STAGE1_BEST_CHECKPOINT, map_location="cpu", weights_only=False)
stage1_state = stage1_payload["model_state_dict"]
current_state = model.state_dict()
compatible_state = {
    key: value for key, value in stage1_state.items()
    if key in current_state
    and current_state[key].shape == value.shape
    and not key.startswith("embedding_adapter.")
}
load_result = model.load_state_dict(compatible_state, strict=False)
loaded_backbone = sum(key.startswith("backbone.") for key in compatible_state)
loaded_classifier = sum(key.startswith("classifier.") for key in compatible_state)
print("Initialized compatible Stage 1 weights.")
print("Loaded backbone tensors:", loaded_backbone)
print("Loaded classifier tensors:", loaded_classifier)
print("Stage 1 adapter weights intentionally not loaded; new 243-dim adapter starts fresh.")

# Freeze the pretrained backbone. Only the adapter and binary head are trainable. Only the adapter and binary head are trainable.
for parameter in model.backbone.parameters():
    parameter.requires_grad = False
for parameter in model.embedding_adapter.parameters():
    parameter.requires_grad = True
for parameter in model.classifier.parameters():
    parameter.requires_grad = True

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
expected_parameters = (2048 * ADAPTER_DIM + ADAPTER_DIM) + (ADAPTER_DIM * 2048 + 2048) + (2048 * 2 + 2)
print(f"Adapter bottleneck dimension: {ADAPTER_DIM}")
print(f"Trainable parameters: {trainable_parameters:,}")
print(f"Expected trainable parameters: {expected_parameters:,}")
assert trainable_parameters == expected_parameters, "Unexpected adapter/head parameter count."
assert trainable_parameters == 1_001_717, "Expected exactly 1,001,717 trainable parameters for Stage 2."


Pretrained checkpoint loaded.
Missing keys: [] 
Unexpected keys: [] 
Initialized compatible Stage 1 weights.
Loaded backbone tensors: 84
Loaded classifier tensors: 2
Stage 1 adapter weights intentionally not loaded; new 243-dim adapter starts fresh.
Adapter bottleneck dimension: 243
Trainable parameters: 1,001,717
Expected trainable parameters: 1,001,717


In [8]:
# 7. Metrics and epoch runner
def compute_metrics(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    hs = (2 * sensitivity * specificity / (sensitivity + specificity)
          if (sensitivity + specificity) else 0.0)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "sensitivity": float(sensitivity),
        "specificity": float(specificity),
        "HS_score": float(hs),
        "confusion_matrix": cm.tolist(),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {"params": model.classifier.parameters(), "lr": CLASSIFIER_LR},
        {"params": model.embedding_adapter.parameters(), "lr": ADAPTER_LR},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=3
)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)


def run_epoch(model, loader, training):
    model.train(training)
    # The whole pretrained backbone is frozen: keep it in eval mode so its
    # BatchNorm running statistics and any backbone dropout behavior stay fixed.
    model.backbone.eval()
                
    total_loss = 0.0
    y_true, y_pred = [], []

    for waveforms, labels in tqdm(loader, leave=False):
        waveforms = waveforms.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            with torch.autocast(device_type=device.type,
                                dtype=torch.float16,
                                enabled=(USE_AMP and device.type == "cuda")
                               ):
                logits = model(waveforms)
            
                # Check model outputs before calculating the loss
                if not torch.isfinite(logits).all():
                    raise RuntimeError(f"Non-finite logits detected. Training={training}")
            
                loss = criterion(logits, labels)
            
            # Check loss BEFORE backward() or optimizer.step()
            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Non-finite loss detected. "
                    f"Training={training}, loss={loss.item()}"
                )


            if training:
                if USE_AMP:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    optimizer.step()


        total_loss += loss.item() * waveforms.size(0)
        predictions = logits.argmax(dim=1)
        y_true.extend(labels.detach().cpu().tolist())
        y_pred.extend(predictions.detach().cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(total_loss / len(loader.dataset))
    return metrics

In [9]:
model.eval()

waveforms, labels = next(iter(train_loader))
waveforms = waveforms.to(device)

with torch.no_grad():
    logits = model(waveforms)

print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Logits shape:", tuple(logits.shape))
print("Logits finite:", torch.isfinite(logits).all().item())

bad_buffers = [
    name for name, buf in model.named_buffers()
    if torch.is_floating_point(buf)
    and not torch.isfinite(buf).all()
]
print("Invalid model buffers:", bad_buffers)

if not torch.isfinite(logits).all():
    raise RuntimeError(
        "Fresh model still produces NaN/Inf logits. Do not start training."
    )

print("Full-precision forward pass passed.")


Waveforms finite: True
Logits shape: (8, 2)
Logits finite: True
Invalid model buffers: []
Full-precision forward pass passed.


In [10]:
batch = next(iter(train_loader))
waveforms, labels = batch[:2]

print("Waveform shape:", waveforms.shape)
print("Waveforms finite:", torch.isfinite(waveforms).all().item())
print("Waveform min/max:", waveforms.min().item(), waveforms.max().item())
print("Labels:", torch.unique(labels))
print("Labels finite:", torch.isfinite(labels).all().item())

Waveform shape: torch.Size([8, 160000])
Waveforms finite: True
Waveform min/max: -1.001936912536621 1.0104199647903442
Labels: tensor([0, 1])
Labels finite: True


In [11]:
# 8. Train and save best/last checkpoints
history = []
best_val_hs = -1.0
best_epoch = None

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, training=True)
    val_metrics = run_epoch(model, val_loader, training=False)
    scheduler.step(val_metrics["HS_score"])
    classifier_lr_now = optimizer.param_groups[0]["lr"]
    adapter_lr_now = optimizer.param_groups[1]["lr"]

    history.append({
        "epoch": epoch,
        "classifier_lr": classifier_lr_now,
        "adapter_lr": adapter_lr_now,
        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],
        "train_sensitivity": train_metrics["sensitivity"],
        "train_specificity": train_metrics["specificity"],
        "train_HS_score": train_metrics["HS_score"],
        "val_loss": val_metrics["loss"],
        "val_accuracy": val_metrics["accuracy"],
        "val_sensitivity": val_metrics["sensitivity"],
        "val_specificity": val_metrics["specificity"],
        "val_HS_score": val_metrics["HS_score"],
    })

    payload = {
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "val_metrics": val_metrics,
        "seed": SEED,
        "class_mapping": {"Normal": 0, "Abnormal": 1},
        "config": {
            "sample_rate": SAMPLE_RATE,
            "segment_seconds": SEGMENT_SECONDS,
            "stage": "stage2_1m_adapter",
            "adapter_dim": ADAPTER_DIM,
            "trainable_parameters": trainable_parameters,
            "classifier_lr": classifier_lr_now,
            "adapter_lr": adapter_lr_now,
            "batch_size": BATCH_SIZE,
        },
    }
    torch.save(payload, LAST_CHECKPOINT)

    if val_metrics["HS_score"] > best_val_hs:
        best_val_hs = val_metrics["HS_score"]
        best_epoch = epoch
        torch.save(payload, BEST_CHECKPOINT)

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"train loss={train_metrics['loss']:.4f}, acc={train_metrics['accuracy']:.4f}, "
        f"HS={train_metrics['HS_score']:.4f} | "
        f"val loss={val_metrics['loss']:.4f}, acc={val_metrics['accuracy']:.4f}, "
        f"SE={val_metrics['sensitivity']:.4f}, SP={val_metrics['specificity']:.4f}, "
        f"HS={val_metrics['HS_score']:.4f} | "
        f"classifier_lr={classifier_lr_now:.2e}, adapter_lr={adapter_lr_now:.2e}"
    )

history_df = pd.DataFrame(history)
display(history_df)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Best epoch:", best_epoch, "| best validation HS:", round(best_val_hs, 4))
print("Best checkpoint:", BEST_CHECKPOINT.resolve())
print("Last checkpoint:", LAST_CHECKPOINT.resolve())

Epoch 01/20 | train loss=0.5557, acc=0.7145, HS=0.7143 | val loss=0.5428, acc=0.7184, SE=0.7199, SP=0.7164, HS=0.7181 | classifier_lr=3.00e-04, adapter_lr=1.00e-04


Epoch 02/20 | train loss=0.5516, acc=0.7202, HS=0.7203 | val loss=0.5510, acc=0.6910, SE=0.8901, SP=0.4145, HS=0.5656 | classifier_lr=3.00e-04, adapter_lr=1.00e-04


Epoch 03/20 | train loss=0.5558, acc=0.7076, HS=0.7071 | val loss=0.5401, acc=0.6971, SE=0.8586, SP=0.4727, HS=0.6098 | classifier_lr=3.00e-04, adapter_lr=1.00e-04


Epoch 04/20 | train loss=0.5487, acc=0.7159, HS=0.7159 | val loss=0.5387, acc=0.7108, SE=0.7618, SP=0.6400, HS=0.6956 | classifier_lr=3.00e-04, adapter_lr=1.00e-04


Epoch 05/20 | train loss=0.5491, acc=0.7277, HS=0.7276 | val loss=0.5375, acc=0.7093, SE=0.7827, SP=0.6073, HS=0.6839 | classifier_lr=1.50e-04, adapter_lr=5.00e-05


...

Epoch 18/20 | train loss=0.5073, acc=0.7481, HS=0.7475 | val loss=0.5674, acc=0.7245, SE=0.6911, SP=0.7709, HS=0.7288 | classifier_lr=3.75e-05, adapter_lr=1.25e-05


Epoch 19/20 | train loss=0.5060, acc=0.7481, HS=0.7476 | val loss=0.5628, acc=0.7230, SE=0.7225, SP=0.7236, HS=0.7231 | classifier_lr=3.75e-05, adapter_lr=1.25e-05


Epoch 20/20 | train loss=0.5048, acc=0.7489, HS=0.7487 | val loss=0.5634, acc=0.7154, SE=0.6780, SP=0.7673, HS=0.7199 | classifier_lr=3.75e-05, adapter_lr=1.25e-05


,epoch,classifier_lr,adapter_lr,train_loss,train_accuracy,train_sensitivity,train_specificity,train_HS_score,val_loss,val_accuracy,val_sensitivity,val_specificity,val_HS_score
0,1,0.000300,0.000100,0.555661,0.714491,0.708898,0.719799,0.714307,0.542836,0.718417,0.719895,0.716364,0.718125
1,2,0.000300,0.000100,0.551601,0.720230,0.723041,0.717562,0.720291,0.551009,0.691020,0.890052,0.414545,0.565641
2,3,0.000300,0.000100,0.555806,0.707604,0.695345,0.719239,0.707090,0.540054,0.697108,0.858639,0.472727,0.609753
3,4,0.000300,0.000100,0.548719,0.715925,0.715969,0.715884,0.715927,0.538717,0.710807,0.761780,0.640000,0.695600
4,5,0.000150,0.000050,0.549052,0.727690,0.724219,0.730984,0.727586,0.537511,0.709285,0.782723,0.607273,0.683925
5,6,0.000150,0.000050,0.541324,0.725681,0.720684,0.730425,0.725522,0.549843,0.730594,0.693717,0.781818,0.735138
6,7,0.000150,0.000050,0.540772,0.723673,0.715380,0.731544,0.723372,0.534437,0.704718,0.782723,0.596364,0.676952
7,8,0.000150,0.000050,0.538946,0.730560,0.739540,0.722036,0.730683,0.559578,0.732116,0.659686,0.832727,0.736175
8,9,0.000150,0.000050,0.534646,0.731133,0.723630,0.738255,0.730869,0.562008,0.736682,0.662304,0.840000,0.740643
9,10,0.000150,0.000050,0.532594,0.731994,0.731291,0.732662,0.731976,0.545796,0.718417,0.714660,0.723636,0.719120


Best epoch: 9 | best validation HS: 0.7406
Best checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_stage2_1m_adapter\cnn14_best_val_hs.pt
Last checkpoint: D:\.JKU CS courses\Master Thesis\models\cnn14_stage2_1m_adapter\cnn14_last_epoch.pt


In [12]:
# 9. Final official test evaluation
# Compare the best Stage 1 and Stage 2 checkpoints on the official test set.


STAGE1_CHECKPOINT = Path(
    "../models/cnn14_stage1_100k_adapter/cnn14_best_val_hs.pt"
)
STAGE2_CHECKPOINT = Path(
    "../models/cnn14_stage2_1m_adapter/cnn14_best_val_hs.pt"
)

def evaluate_saved_checkpoint(checkpoint_path, adapter_dim, stage_name):
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"{stage_name} checkpoint not found: "
            f"{checkpoint_path.resolve()}"
        )

    # Reconstruct the correct architecture for this stage.
    eval_backbone = Cnn14(
        sample_rate=16000,
        window_size=1024,
        hop_size=320,
        mel_bins=64,
        fmin=50,
        fmax=8000,
        classes_num=527,
    )

    eval_model = CNN14BinaryClassifier(
        eval_backbone,
        adapter_dim=adapter_dim,
    ).to(device)

    # Load the complete trained model, including the adapter and classifier.
    payload = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    eval_model.load_state_dict(payload["model_state_dict"], strict=True)
    eval_model.eval()

    total_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for waveforms, labels in tqdm(test_loader, leave=False):
            waveforms = waveforms.to(device)
            labels = labels.to(device)

            logits = eval_model(waveforms)
            loss = criterion(logits, labels)

            if not torch.isfinite(logits).all():
                raise RuntimeError(
                    f"{stage_name} produced non-finite logits."
                )

            total_loss += loss.item() * waveforms.size(0)
            y_true.extend(labels.cpu().tolist())
            y_pred.extend(logits.argmax(dim=1).cpu().tolist())

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / len(test_loader.dataset)

    result = {
        "stage": stage_name,
        "adapter_dim": adapter_dim,
        "trainable_parameters": (
            100_377 if adapter_dim == 23 else 1_001_717
        ),
        "checkpoint_epoch": payload["epoch"],
        "test_loss": metrics["loss"],
        "test_accuracy": metrics["accuracy"],
        "test_sensitivity": metrics["sensitivity"],
        "test_specificity": metrics["specificity"],
        "test_HS_score": metrics["HS_score"],
        "confusion_matrix": metrics["confusion_matrix"],
    }

    print(f"\n{stage_name}")
    print(f"Checkpoint epoch: {payload['epoch']}")
    print(f"Test loss:        {metrics['loss']:.4f}")
    print(f"Test accuracy:    {metrics['accuracy']:.4f}")
    print(f"Sensitivity:      {metrics['sensitivity']:.4f}")
    print(f"Specificity:      {metrics['specificity']:.4f}")
    print(f"HS-score:         {metrics['HS_score']:.4f}")
    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(np.array(metrics["confusion_matrix"]))

    return result


# Evaluate the best validation-HS checkpoint from each stage.
stage1_result = evaluate_saved_checkpoint(
    STAGE1_CHECKPOINT, adapter_dim=23, stage_name="Stage 1 (100k)"
)

stage2_result = evaluate_saved_checkpoint(
    STAGE2_CHECKPOINT, adapter_dim=243, stage_name="Stage 2 (1M)"
)

comparison = pd.DataFrame([stage1_result, stage2_result])

display(comparison[
    [
        "stage",
        "trainable_parameters",
        "checkpoint_epoch",
        "test_loss",
        "test_accuracy",
        "test_sensitivity",
        "test_specificity",
        "test_HS_score",
    ]
].round(4))

# Save the comparison results separately from the training outputs.
comparison_path = Path("../models/cnn14_stage_comparison_test.csv")
comparison.drop(columns=["confusion_matrix"]).to_csv(
    comparison_path, index=False
)
print("Comparison saved to:", comparison_path.resolve())



Stage 1 (100k)
Checkpoint epoch: 11
Test loss:        0.7342
Test accuracy:    0.5885
Sensitivity:      0.5234
Specificity:      0.6371
HS-score:         0.5747
Confusion matrix [[TN, FP], [FN, TP]]:
[[1006  573]
 [ 561  616]]



Stage 2 (1M)
Checkpoint epoch: 9
Test loss:        0.7311
Test accuracy:    0.5954
Sensitivity:      0.4681
Specificity:      0.6903
HS-score:         0.5579
Confusion matrix [[TN, FP], [FN, TP]]:
[[1090  489]
 [ 626  551]]


,stage,trainable_parameters,checkpoint_epoch,test_loss,test_accuracy,test_sensitivity,test_specificity,test_HS_score
0,Stage 1 (100k),100377,11,0.7342,0.5885,0.5234,0.6371,0.5747
1,Stage 2 (1M),1001717,9,0.7311,0.5954,0.4681,0.6903,0.5579


Comparison saved to: D:\.JKU CS courses\Master Thesis\models\cnn14_stage_comparison_test.csv
